# Experiments Analysis & Hypothesis Notebook

This notebook serves as the central collaboration space for hypothesis testing and experiment analysis.

## How to Use This Notebook

1. **Manifest (auto-generated)**: `experiments/experiments_manifest.csv` is automatically updated when evaluations run
2. **Hypotheses (human-defined)**: Define hypotheses below with linked experiments
3. **Analysis (collaborative)**: Both human and AI can add observations and analysis

## Workflow
1. Run evaluations: `python notebooks/eval_fun.py` → updates manifest
2. Load manifest below to see all evaluated experiments
3. Define or update hypotheses with experiment IDs
4. Analyze results collaboratively

In [1]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from os.path import join, exists
import json

# Load manifest utilities
from eval_fun import load_experiments_manifest, update_experiments_manifest, MANIFEST_PATH

# Plotting settings
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 6)

---
# 1. Experiments Manifest

Auto-generated table of all evaluated experiments.

In [3]:
# Load the experiments manifest
manifest = load_experiments_manifest()

if len(manifest) > 0:
    # Display summary
    print(f"Total experiments: {len(manifest)}")
    print(f"Datasets: {manifest['dataset'].unique().tolist()}")
    print(f"Architectures: {manifest['architecture'].unique().tolist()}")
    print()
    
    # Show table
    # display(manifest[[
    #     "exp_id", "dataset", "architecture", "attention", "cross_attention", 
    #     "best_val_loss", "best_test_r2", "last_evaluated"]])
    
    display(manifest)
else:
    print("No experiments in manifest yet. Run evaluations to populate.")

Total experiments: 25
Datasets: ['scm6', 'scm7']
Architectures: ['SingleCausalLayer']



,exp_id,path,dataset,architecture,attention,cross_attention,max_epochs,learning_rate,batch_size,optimizer,...,soft_hamming_cross_mean,soft_hamming_cross_worst,soft_hamming_self_best,soft_hamming_self_mean,soft_hamming_self_worst,dag_source,final_hsic_mean,final_hsic_std,last_evaluated,available_evals
0,single_Lie_CC_scm6_54803384,C:\Users\ScipioneFrancesco\Documents\Projects\...,scm6,SingleCausalLayer,LieAttention,CausalCrossAttention,10,0.0005,64,adamw,...,0.493837,0.502386,0.477709,0.495420,0.521956,phi,NaN,NaN,2026-02-09T15:20:41,"[""eval_attention_scores"", ""eval_do"", ""eval_emb..."
1,single_Lie_CC_scm6_54834710,C:\Users\ScipioneFrancesco\Documents\Projects\...,scm6,SingleCausalLayer,LieAttention,CausalCrossAttention,100,0.0005,64,adamw,...,0.496642,0.509771,0.358411,0.501314,0.641868,phi,NaN,NaN,2026-02-09T17:44:06,"[""eval_attention_scores"", ""eval_do"", ""eval_emb..."
2,single_Lie_CC_scm6_54916195,C:\Users\ScipioneFrancesco\Documents\Projects\...,scm6,SingleCausalLayer,LieAttention,CausalCrossAttention,100,0.0050,64,adamw,...,0.494069,0.500385,0.397192,0.483139,0.529593,phi,NaN,NaN,2026-02-09T20:21:02,"[""eval_attention_scores"", ""eval_do"", ""eval_emb..."
3,single_Lie_CC_scm6_54946595,C:\Users\ScipioneFrancesco\Documents\Projects\...,scm6,SingleCausalLayer,LieAttention,CausalCrossAttention,10,0.0005,64,adamw,...,0.493837,0.502386,0.477709,0.495420,0.521956,phi,NaN,NaN,2026-02-09T22:37:19,"[""eval_attention_scores"", ""eval_do"", ""eval_emb..."
4,single_Lie_CC_scm6_55015699,C:\Users\ScipioneFrancesco\Documents\Projects\...,scm6,SingleCausalLayer,LieAttention,CausalCrossAttention,100,0.0010,64,adamw,...,0.489267,0.500320,0.479437,0.567484,0.625779,phi,0.000071,0.000069,2026-02-10T00:46:29,"[""eval_attention_scores"", ""eval_do"", ""eval_emb..."
5,single_Lie_CC_scm7_55058272,C:\Users\ScipioneFrancesco\Documents\Projects\...,scm7,SingleCausalLayer,LieAttention,CausalCrossAttention,100,0.0010,64,adamw,...,0.506592,0.518462,0.478025,0.507074,0.523079,phi,0.000012,0.000009,2026-02-10T02:53:31,"[""eval_attention_scores"", ""eval_do"", ""eval_emb..."
6,single_Lie_PhiSM_scm6_54803355,C:\Users\ScipioneFrancesco\Documents\Projects\...,scm6,SingleCausalLayer,LieAttention,PhiSoftMax,10,0.0005,64,adamw,...,0.482615,0.487368,0.478817,0.505985,0.529213,phi,NaN,NaN,2026-02-10T11:04:25,"[""eval_attention_scores"", ""eval_do"", ""eval_emb..."
7,single_Lie_PhiSM_scm6_54834824,C:\Users\ScipioneFrancesco\Documents\Projects\...,scm6,SingleCausalLayer,LieAttention,PhiSoftMax,100,0.0005,64,adamw,...,0.485241,0.502221,0.497425,0.508445,0.522198,phi,NaN,NaN,2026-02-10T13:24:09,"[""eval_attention_scores"", ""eval_do"", ""eval_emb..."
8,single_Lie_PhiSM_scm6_55003837,C:\Users\ScipioneFrancesco\Documents\Projects\...,scm6,SingleCausalLayer,LieAttention,PhiSoftMax,100,0.0010,64,adamw,...,0.477082,0.482184,0.521232,0.523528,0.529858,phi,NaN,NaN,2026-02-10T16:10:12,"[""eval_attention_scores"", ""eval_do"", ""eval_emb..."
9,single_Lie_PhiSM_scm7_55058561,C:\Users\ScipioneFrancesco\Documents\Projects\...,scm7,SingleCausalLayer,LieAttention,PhiSoftMax,100,0.0010,64,adamw,...,0.518365,0.557574,0.477254,0.499971,0.537156,phi,NaN,NaN,2026-02-10T18:45:40,"[""eval_attention_scores"", ""eval_do"", ""eval_emb..."


---
# 2. Hypotheses

Define scientific hypotheses and link them to experiments by `exp_id`.

## H1: Intervention Invariance is guaranteed when the true DAG is provided

**Question:** Does the transformer prevent non-causal mixing if the actual causal DAG is provided in the attention blocks?

**Expected Outcomes:**
- Variations in non-parents do not change children's distribution. For scm6: interventions in S1 do not change X2 and its descendent's distributions.

**Experiments:**

**Status:** The hypothesis is confirmed.

In [ ]:
# H1: Intervention Invariance is guaranteed when the true DAG is provided
H1_experiments = [
    # "single_Lie_CC_scm6_loc001",  # TODO add experiments
    # Add more experiment IDs here as they are run
]

# Filter manifest for H1 experiments
if len(manifest) > 0:
    h1_data = manifest[manifest["exp_id"].isin(H1_experiments)]
    if len(h1_data) > 0:
        display(h1_data)
    else:
        print("No H1 experiments found in manifest yet.")

### H1 Observations

<!-- Human observations -->
**Francesco (date):**
- [Add observations here]

<!-- AI observations -->
**Cline (date):**
- [AI analysis will be added here]

## H2: Residual - Parents independence higher if true DAG is provided, compared to when the wrong one is provided instead.

**Question:** Consider the Hilber-Schmidt Independence Criterion (HSIC) between residuals and sources, do we measure lower HSIC when we provide the true DAG compared to a wrong one?

**Expected Outcomes:**
- HSIC lower when the true DAG is provided

**Experiments:**

**Status:** The hypothesis is confirmed.

In [ ]:
# H2: Residual - Parents independence higher if true DAG is provided, compared to when the wrong one is provided instead.
H2_experiments = [
    # "single_Lie_CC_scm6_loc001",  # TODO add experiments
    # Add more experiment IDs here as they are run
]

# Filter manifest for H2 experiments
if len(manifest) > 0:
    h2_data = manifest[manifest["exp_id"].isin(H2_experiments)]
    if len(h2_data) > 0:
        display(h2_data)
    else:
        print("No H2 experiments found in manifest yet.")

### H2 Observations

<!-- Human observations -->
**Francesco (date):**
- Overall, H2 seems to hold
- The HSIC seems also depending on the quality of the fit: even when the true DAG is provided, if the model has insufficient capacity to fit the data, it results in a high HSIC.
- H2 validation suggests that HSIC could be potentially used as a regularization term to improve causal discovery. 

<!-- AI observations -->
**Cline (date):**
- [AI analysis will be added here]

## H3: Lie attention improves causal discovery

**Question:** Does Lie attention improve the transformer capability in distinguishing between correlation and causality?

**Expected Outcomes:**
- Hamming distance lower in models with Lie attention

**Experiments:**

**Status:** Work in progress

In [ ]:
# H3: Lie attention improves causal discovery
H3_experiments = [
    # "single_Lie_CC_scm6_loc001",  # TODO add experiments
    # Add more experiment IDs here as they are run
]

# Filter manifest for H2 experiments
if len(manifest) > 0:
    h3_data = manifest[manifest["exp_id"].isin(H3_experiments)]
    if len(h3_data) > 0:
        display(h3_data)
    else:
        print("No H3 experiments found in manifest yet.")

### H3 Observations

<!-- Human observations -->
**Francesco (date):**
 

<!-- AI observations -->
**Cline (date):**
- [AI analysis will be added here]

## H4: Freezing sequential stage parameters improves causal discovery

**Question:**

**Expected Outcomes:**

**Experiments:**

**Status:** Not started

## H5: Reducing the batch size improves causal discovery

**Question:**

**Expected Outcomes:**

**Experiments:**

**Status:** Not started

## H6: Embedding separation improves causal discovery

**Question:**

**Expected Outcomes:**

**Experiments:**

**Status:** Not started

## H7: Learning stage noise improves causal discovery

**Question:** By introducing a learnable noise at each stage, can the stage output be better explained by inter-stage interactions (self-attention) than short-cutting to stage parents (cross-attention)?

*Example:* In scm6, X1 could learn X2 from its parents S2 and S3 and completely ignore the causal relation X2->X1. But, if we equip X2 with a stochastic noise (learnable), X1 could prefer X2 from its deterministic parents, as this noise could better explain the targets. 

**Expected Outcomes:**

**Experiments:**

**Status:** Not started

## H8: DAGs from non-Linear SCM are easier to learn from linear ones

**Question:**

**Expected Outcomes:**

**Experiments:**

**Status:** Not started

## H9: The HSIC regularizer helps the transformer to converge to the true DAG representation. 

**Question:**

**Expected Outcomes:**

**Experiments:**

**Status:** Not started

## H10: The KL regularizer helps the transformer to converge to the true DAG representation. 

**Question:**

**Expected Outcomes:**

**Experiments:**

**Status:** Not started

---
# 3. Cross-Hypothesis Comparisons

Analysis code for comparing experiments across hypotheses.

In [ ]:
# Example: Compare test R² across attention types
if len(manifest) > 0 and 'attention' in manifest.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    if manifest['best_test_r2'].notna().any():
        sns.boxplot(data=manifest, x='attention', y='best_test_r2', ax=ax)
        ax.set_title('Test R² by Attention Type')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("No test R² data available yet.")

---
# 4. Notes & TODO

## Open Questions
- Why different splits of the same dataset lead to very different learning dynamics? Is there a metric we can use to determine when learning from a particular dataset is "hard"?
- [ ] Add question 2

## Next Experiments to Run
- [ ] Experiment idea 1
- [ ] Experiment idea 2

## Conclusions
- [Summary of findings will be added here]